In [ ]:
#test_landsat_histogram creates and plots histogram of rgb values. 
#Illustrates difference between scaled and unscaled images.
import ee
import geemap
import matplotlib.pyplot as plt
import numpy as np

# Initialize Earth Engine (authenticate if needed)
try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()

In [ ]:
# Step 1: Define a region of interest (ROI) - example: San Francisco area
roi = ee.Geometry.Rectangle([-122.5, 37.7, -122.3, 37.9])  # [lon_min, lat_min, lon_max, lat_max]

# Step 2: Load and filter Landsat 8 SR collection
collection = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')  # Surface Reflectance, Tier 1
              .filterBounds(roi)
              .filterDate('2014-03-01', '2014-05-31')  # Summer 2020 for low clouds
              .filterMetadata('CLOUD_COVER', 'less_than', 10)  # Low cloud cover
              .sort('CLOUD_COVER')  # Sort by cloud cover
              .first())  # Select the first (clearest) image. Use .median() for a composite instead.

print(f'Selected image ID: {collection.get("system:index").getInfo()}')
#print(f'Collection size: {collection.size().getInfo()}') #error since collection is only 1 image (.first arguement)
#known good image::: image = ee.Image('LANDSAT/LC08/C02/T1_TOA/LC08_044034_20140318')

# Step 3: Select RGB bands and apply scaling (multiply by 0.0000275 and add -0.2 for reflectance)
rgb_bands = ['SR_B4', 'SR_B3', 'SR_B2']  # Red, Green, Blue
rgb_image = collection.select(rgb_bands).multiply(0.0000275).add(-0.2)
rgb_image = rgb_image.updateMask(rgb_image.gte(0))  # Mask negative values (artifacts)

In [ ]:
# Step 4: Create an interactive map for visualization (optional)
m = geemap.Map(center=[37.8, -122.4], zoom=11)
vis_params = {'bands': rgb_bands, 'min': 0, 'max': 0.3, 'gamma': 1.2}  # Quick RGB viz (unscaled)
m.addLayer(collection.clip(roi), {}, 'Landsat RGB') #doesn't work with vis_params (display is all white)
m.addLayer(rgb_image,vis_params,'rgb') #does work with vis_params
#m.addLayerControl()
m  # Displays the map in Jupyter/Colab

In [ ]:
# Step 5: Sample pixels to NumPy array for local histogram plotting
# Use a region slightly larger than ROI for better sampling; scale=30m for Landsat resolution
array = geemap.ee_to_numpy(rgb_image, region=roi.buffer(1000), scale=30) 
#, default_value=0) #Exception: Invalid JSON payload received. Unknown name "default_value": Cannot find field.

# Flatten the array to 1D (exclude nodata/masked values)
red = array[:,:,0].flatten()
green = array[:,:,1].flatten()
blue = array[:,:,2].flatten()

# Remove nodata values (0 or NaN)
valid_mask = (red > 0) & (green > 0) & (blue > 0) & (~np.isnan(red)) & (~np.isnan(green)) & (~np.isnan(blue))
red = red[valid_mask]
green = green[valid_mask]
blue = blue[valid_mask]

print(f'Sampled {len(red)} valid pixels')

In [ ]:
# Step 6: Plot histograms
plt.figure(figsize=(10, 6))
plt.hist(red, bins=50, alpha=0.7, label='Red (SR_B4)', color='red', density=True)
plt.hist(green, bins=50, alpha=0.7, label='Green (SR_B3)', color='green', density=True)
plt.hist(blue, bins=50, alpha=0.7, label='Blue (SR_B2)', color='blue', density=True)
plt.xlabel('Reflectance Value')
plt.ylabel('Density')
plt.title('Histogram of RGB Values from Landsat 8 Image')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
#now for unscaled version
# Step 5: Sample pixels to NumPy array for local histogram plotting
# Use a region slightly larger than ROI for better sampling; scale=30m for Landsat resolution
array = geemap.ee_to_numpy(collection, region=roi.buffer(1000), scale=30) 
#, default_value=0) #Exception: Invalid JSON payload received. Unknown name "default_value": Cannot find field.

# Flatten the array to 1D (exclude nodata/masked values)
red = array[:,:,0].flatten()
green = array[:,:,1].flatten()
blue = array[:,:,2].flatten()

# Remove nodata values (0 or NaN)
valid_mask = (red > 0) & (green > 0) & (blue > 0) & (~np.isnan(red)) & (~np.isnan(green)) & (~np.isnan(blue))
red = red[valid_mask]
green = green[valid_mask]
blue = blue[valid_mask]

print(f'Sampled {len(red)} valid pixels')

In [ ]:
# Step 6: Plot histograms
plt.figure(figsize=(10, 6))
plt.hist(red, bins=100, alpha=0.7, label='Red (SR_B4)', color='red', density=True)
plt.hist(green, bins=100, alpha=0.7, label='Green (SR_B3)', color='green', density=True)
plt.hist(blue, bins=100, alpha=0.7, label='Blue (SR_B2)', color='blue', density=True)
plt.xlabel('Reflectance Value')
plt.ylabel('Density')
plt.title('Histogram of RGB Values from Landsat 8 Image')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()